# Pipeline di Classificazione Curriculum Vitae (Resume)

Questo notebook implementa una pipeline di Machine Learning per classificare i CV in categorie.

**Step:**
1. Caricamento e Pulizia Dati
2. Split del Dataset (Train / Validation / Test)
3. Feature Extraction (TF-IDF)
4. Addestramento Modello (Linear SVM)
5. Valutazione
6. Inferenza

In [1]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import re
import seaborn as sns
import string
from wordcloud import WordCloud
import nltk
%load_ext autoreload
%autoreload 2
from tools import utils as u, plotting as up

## 1. Caricamento e Preprocessing dei Dati

In [2]:
df = pd.read_csv('data/Resume.csv')

df['text'] = df['Resume_str'].apply(u.preprocess_text)

# Visualizzazione delle classi
print(f"Dataset caricato con successo: {df.shape[0]} righe, {df.shape[1]} colonne")
print("\nDistribuzione delle Categorie:")
print(df['Category'].value_counts())

Dataset caricato con successo: 2484 righe, 5 colonne

Distribuzione delle Categorie:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


# Wordcloud

In [ ]:
MAX_WORDS = 500

wc = WordCloud(
    width=800, 
    height=700, 
    background_color='white',
    max_words=MAX_WORDS,
    contour_width=0,
    colormap='viridis'  # 'viridis', 'plasma', or 'inferno' work well for data
)

fig = up.wordcloud(wc, df["text"], MAX_WORDS, ENGLISH_STOP_WORDS)
fig.savefig("data/wordcloud/wordcloud_overall.png", dpi=300)

for category in df['Category'].unique():
    category_df = df[df['Category'] == category]
    fig = up.wordcloud(wc, category_df["text"], MAX_WORDS, ENGLISH_STOP_WORDS, title="Word Cloud for Category: {category}")
    fig.savefig(f"data/wordcloud/wordcloud_{category}.png", dpi=300)


In [ ]:
counts = df['Category'].value_counts().reset_index()
counts.columns = ['Category','Count']
fig = px.bar(counts, x='Category', y='Count', title='Distribuzione delle Categorie', height=500 ,width=1200)
fig.show()

## 2. Suddivisione del Dataset (Train + Validation + Test)

Suddividiamo i dati in:
- **Train Set (70%)**: Per addestrare il modello.
- **Validation Set (15%)**: Per il tuning degli iperparametri (implicito in questa pipeline semplificata).
- **Test Set (15%)**: Per la valutazione finale imparziale.

In [3]:
# Step 1: preprocessing (è già stato fatto nella wordcloud)
# df["text"] = df["Resume_str"].apply(u.preprocess_text)
# df["text"].sample(10)

X = df['text']
y = df['Category']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

print(f"Dimensioni Train: {X_train.shape[0]}")
print(f"Dimensioni Validation: {X_val.shape[0]}")
print(f"Dimensioni Test: {X_test.shape[0]}")

Dimensioni Train: 1738
Dimensioni Validation: 373
Dimensioni Test: 373


## 3. Feature Extraction
Esistono due principali metodologie di Feature Extraction:
- Bag of words
- TF-IDF

In [4]:
# Step 1: preprocessing (è già stato fatto nella wordcloud)
# df["text"] = df["Resume_str"].apply(u.preprocess_text)
# df["text"].sample(10)

# Step 2: parole univoche (solo per dimostrazione)
unique_words = u.unique_text(df["text"])
print(len(unique_words))
wordsdf = pd.DataFrame(unique_words)

word2count = {} 
for data in df["text"]: 
    words = nltk.word_tokenize(data) 
    for word in words: 
        if word not in word2count.keys(): 
            word2count[word] = 1
        else: 
            word2count[word] += 1


55030


## TF-IDF
Utilizziamo **TF-IDF (Term Frequency - Inverse Document Frequency)**. 
È molto efficace per il testo perché penalizza le parole troppo comuni (stop words) e valorizza quelle distintive per ogni categoria.
TF-IDF è il prodotto di TF e IDF, dove la prima è alta per un termine che si ripete tante volte all'interno di un documento e la seconda, invece, è alta se lo stesso termine si ripete raramente tra tutti i documenti. In questo caso signfica che il termine ha molta rilevanza per categorizzare un documento e distinguerlo da altri.

In [15]:
# Inizializzazione del vettorizzatore
# max_features=5000 limita il vocabolario alle 5000 parole più importanti per ridurre la dimensionalità
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 3))

X = vectorizer.fit_transform(df["text"])
y = df["Category"]

print(f"Shape della matrice di feature (Train): {X.shape}")

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

Shape della matrice di feature (Train): (2484, 5000)


## 4. Addestramento del Classificatore
Utilizziamo una **Linear SVC (Support Vector Classifier)**, nota per essere veloce e accurata nella classificazione di testi.

In [ ]:
model = LinearSVC(random_state=42, dual='auto')
model.fit(X_train_vec, y_train)
print("Addestramento completato.")

In [16]:
# Naive Bayes
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## 5. Valutazione del Modello
Valutiamo le performance sul **Test Set**.

In [17]:
y_pred = model.predict(X_test)

print("--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred))

print(f"Accuracy finale: {accuracy_score(y_test, y_pred):.4f}")

--- Classification Report (Test Set) ---
                        precision    recall  f1-score   support

            ACCOUNTANT       0.44      0.82      0.57        17
              ADVOCATE       0.43      0.50      0.46        18
           AGRICULTURE       0.00      0.00      0.00        10
               APPAREL       0.75      0.20      0.32        15
                  ARTS       0.50      0.06      0.11        16
            AUTOMOBILE       0.00      0.00      0.00         5
              AVIATION       0.62      0.56      0.59        18
               BANKING       0.50      0.47      0.48        17
                   BPO       0.00      0.00      0.00         4
  BUSINESS-DEVELOPMENT       0.32      0.67      0.44        18
                  CHEF       0.84      0.94      0.89        17
          CONSTRUCTION       0.65      0.65      0.65        17
            CONSULTANT       0.00      0.00      0.00        17
              DESIGNER       0.80      0.50      0.62        1

/home/began/srcs/data_science/DataScience/Progetto_NLP/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/began/srcs/data_science/DataScience/Progetto_NLP/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/began/srcs/data_science/DataScience/Progetto_NLP/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` 

In [18]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig = px.imshow(cm, x=labels, y=labels, color_continuous_scale='Blues',
                labels=dict(x='Predetto', y='Reale', color='Count'),
                text_auto=True, title='Matrice di Confusione', height=800, width=800)
fig.update_layout(xaxis_title='Predetto', yaxis_title='Reale')
fig.show()

## 6. Inferenza
Utilizziamo il modello addestrato su un nuovo testo mai visto.

In [ ]:
sr = df.sample(1).iloc[0]
text = sr['text']
truth_label = sr['Category']
del sr

svec = vectorizer.transform([text])
prediction = model.predict(text)[0]

print(f"Testo Input:\n{text.strip()}")
print(f"\nCategoria Predetta: {prediction}")
print(f"Categoria Reale: {truth_label}")

KeyError: 'cleaned_resume'